# Exercise 2. Classify Text with BOW and TF-IDF
In this exercise, we'll classify text, vectorized with either `Bag-of-Words`(`BOW`) or `TF-IDF`. To recap, `BOW` is a simple count of each `term` (or word) in each text document, represented as document vectors:
```{figure} ../figures/class2/bow.png
---
name: bow
---
Figure from [Zhou (2019)](https://victorzhou.com/blog/bag-of-words/).
```

TF-IDF is like BOW but assigns each word a weighted score instead of a simple count. Words that appear frequently in a document but rarely across all documents get a higher weight, while very common words get lower weights. See also [geeksforgeeks.org](https://www.geeksforgeeks.org/machine-learning/understanding-tf-idf-term-frequency-inverse-document-frequency/).


```{admonition} LLM FRAMING: Why BOW/TF-IDF still matter in the age of LLMs!
:class: dropdown, fuchsia
There are many reasons to still care about classifiers relying on BOW and TF-IDF:
1. Classifiers using BOW/TF-IDF tend to perform relatively well (sometimes better), compared to a LLM-based classifier (Like [BERT](https://www.tensorflow.org/text/tutorials/classify_text_with_bert))
2. It is far cheaper computationally to train than fine-tuning an LLM. 
3. Even when fine-tuning makes sense, we need baselines to measure progress, and BOW/TF-IDF are good for this! 

Our key question should be: Is an LLM-approach worth it if a simpler baseline works just as well?
```

In [505]:
## CODE CHUNK REMOVED FOR USERS - HERE TO RELOAD DATA ##
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

## LOAD DATA ## 
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

raw_df = pd.read_csv(data_path)

## SUBSET DATA ##
df = raw_df[raw_df["model"].isin(["human", "cohere"])]

df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

## SPLIT DATA ##
train_df, val_df= train_test_split(
                                                    df,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df["is_human"]
                                                    )

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_21270/1245980808.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)


## 2.1 Vectorize Text with BOW
Import a `CountVectorizer` object at the top of your notebook:

In [506]:
from sklearn.feature_extraction.text import CountVectorizer

Instantiate a CountVectorizer object. We set `lowercase = True` as we did not do this preprocessing ourselves :). We also limit our amount of features to only the top 500 features to reduce the computational costs:

In [507]:
vectorizer = CountVectorizer(lowercase=True, max_features=500)

We select our text column `"generation"` from our training split. We fit the `vectorizer` and transform our text with `.fit_transform`:

In [508]:
X_train_bow = vectorizer.fit_transform(train_df["generation"])

For our validation split, we'll use `transform` method (we only `fit` our vectorizer to training data!)!

In [509]:
X_val_bow = vectorizer.transform(val_df["generation"])

Let's print the first few features and the first vector

In [510]:
print("Features:", vectorizer.get_feature_names_out()[:30])

print("\nTraining set:")
print(X_train_bow.toarray()[0])

Features: ['000' '10' '12' '15' '20' '30' 'able' 'about' 'action' 'add' 'added'
 'after' 'again' 'against' 'age' 'all' 'almost' 'along' 'also' 'always'
 'am' 'american' 'an' 'and' 'another' 'any' 'anything' 'approach' 'are'
 'around']

Training set:
[0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 5 0 0 0 0 0 0 0 0 0 0 0 1 1
 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 3 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 1 0 0 2 0 0 0 0 3 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2
 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 5 1 0 0 1 0 2 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 2 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 2 2 1 0 2 0 0 0 0 0 0 0 0 0 0 0
 0 1 2 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0


As you can see, there are many features that are aren't present (the many 0's).

## 2.2 Vectorize Text with TF-IDF
For TF-IDF, we can use the same steps as above except for the `vectorizer` being a new one:

In [511]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(lowercase=True, max_features=500)

### Your Turn: Defining a Vectorization Function!
We want to examine whether using `bow` or `tf-idf` affects the classification accuracy. For this purpose, a function could streamline our workflow!

:::{admonition} HANDS-ON
:class: red
Define a function called `vectorize` that:
1. Takes `X_train` and `X_val` as text columns (e.g., `X_train = train_df["generations"]`).
2. Accepts `vec_type`, where you can choose either `"bow"` or `"tf-idf"` and it will select `CountVectorizer` and `TfidfVectorizer` respectively. 
    - You can do this with an `if` and `elif`statement
3. Includes`max_features`parameter to control the vectorizer's`max_features` (like we have done above).
4. Returns `X_train_vectorized` and `X_val_vectorized` 

When you have your function, run it once to create `X_train_bow`and `X_val_bow` and a second time to create `X_train_tfidf`, `X_val_tfidf`
:::

:::{admonition} HINT: `if` and `elif`? What do you mean!
:class: tip, dropdown
Remember how we can check a condition with `if`?

Let's say we want to print all banned colors!
```python
all_colors = ["red", "green", "blue", "yellow", "purple"]
banned_colors = ["red", "purple"]

# for each colour in the list called colours
for color in all_colors:
    if color in banned_colors: #  "in" checks if color is in banned_colors
        print(color + " is a BANNED color") # prints for red and purple
```

But we also want to print `yellow` because it is BRIGHT! We can do this with `elif`:

```python
all_colors = ["red", "green", "blue", "yellow", "purple"]
banned_colors = ["red", "purple"]

# for each colour in the list called colours
for color in all_colors:
    if color in banned_colors: #  "in" checks if color is in banned_colors
        print(color + " is a BANNED color") # prints for red and purple
    elif color == "yellow": 
        print(color + " is a BRIGHT color")
```

**BONUS**: If we want to catch everything that does not match either one of the conditions defined by `if` and `elif`, we can write `else` as the final statement!
```python
all_colors = ["red", "green", "blue", "yellow", "purple"]
banned_colors = ["red", "purple"]

# for each colour in the list called colours
for color in all_colors:
    if color in banned_colors: #  "in" checks if color is in banned_colors
        print(color + " is a BANNED color") # prints for red and purple
    elif color == "yellow": 
        print(color + " is a BRIGHT color")
    else: 
        # if none of the above
        print(color + " is allowed")  # prints for green and blue
```

:::

### Solution
You can check the solution here:

In [512]:
from typing import Literal 
# using Literal is not strictly necessary, 
# but is a way to define the options that you can use for the function!

def vectorize(X_train: pd.Series, X_val: pd.Series, vec_type:Literal["bow", "tf-idf"], max_features:int=500):
    """
    Function to vectorize train and val data! 
    """
    if vec_type == "bow":
        vectorizer = CountVectorizer(lowercase=True, max_features=max_features)
    elif vec_type== "tf-idf":
        vectorizer = TfidfVectorizer(lowercase=True, max_features=max_features)
    else: 
        # this is good code practice, but if your function has no 'else' statement, this is also fine!
        raise ValueError(f"Invalid vec_type: {vec_type}. Must be either 'bow' or 'tf-idf")
    
    X_train_vectorized = vectorizer.fit_transform(X_train)
    X_val_vectorized = vectorizer.transform(X_val)

    return X_train_vectorized, X_val_vectorized


# APPLY TO GET TF-IDF
X_train_tfidf, X_val_tfidf = vectorize(
                        X_train = train_df["generation"], 
                          X_val = val_df["generation"],
                          vec_type="tf-idf"
                          )

## 2.3 Classification
We are ready to do our binary classification (`human` versus `cohere`) using a simple logistic regression. Let's import it from `scikit-learn`:

In [513]:
from sklearn.linear_model import LogisticRegression

> We could also technically use a `Naive Bayes` classifier, using [GaussianNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) for BOW and [MultinomialNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB) for TF-IDF.

Like with the `Countervectorizer`, we can instantiate `LogisticRegression` as our classifier `clf_1`:

In [514]:
# clf is common naming convention
clf_1 = LogisticRegression(
    random_state=42,
    solver="liblinear",   # better for small/medium sparse datasets
    max_iter=1000,        # allow more iterations
    C=1.0,                # adjust if needed (smaller values = stronger regularization)
)

Let's extract our numerical labels as Y:

In [515]:
y_train = train_df["is_human"]

Let's fit our classifier on our vectorized text `X_train_bow` and our `y_train`:

In [516]:
clf_1.fit(X_train_bow, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


## 2.4 Evaluation
Now that we have fit our classifier, we should evaluate it's performance on `X_val_BOW` and `y_val`. Start by importing `classification_report` from scikit-learn (at the top of your notebook):

In [517]:
from sklearn.metrics import classification_report

Let's extract predicted labels `y`:

In [518]:
y_pred = clf_1.predict(X_val_bow)

Now, we can compare to our actual labels `y` val:

In [519]:
y_val = val_df["is_human"]
report = classification_report(y_val, y_pred) 

In [520]:
print(report)

              precision    recall  f1-score   support

           0       0.80      0.90      0.85      5349
           1       0.74      0.55      0.63      2674

    accuracy                           0.78      8023
   macro avg       0.77      0.73      0.74      8023
weighted avg       0.78      0.78      0.78      8023



```{admonition} QUESTION
:class: red

Look at the results. Can you identify any interesting patterns? Why may there be a difference between the raw precision scores and the macro avg. / weighted avg. scores? 

*Remember that 1 equals `human`!*

<details>
<summary>HINT</summary>
Since our classes aren't balanced in any way, the raw accuracy, f1, precision/recall metrics become unreliable. In our case, we would prefer to look at the macro avg or weighted avg! 
</details>
```

### Your Turn: Try with TF-IDF
:::{admonition} HANDS-ON
:class: red
Try to run the classification + evaluation workflow above with the TF-IDF vectorized `X` and see if that changes results!
- You can just replace `X_train_bow` with `X_train_tfidf` (which you returned from your `vectorize` function)
- Make sure you take a second to discuss the results with a friend!
:::

Solution below:

In [521]:
## NO NEED TO IMPORT AGAIN IF YOU DID: 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# fit 
clf_2 = LogisticRegression(
    random_state=42,
    solver="liblinear",   # better for small/medium sparse datasets
    max_iter=1000,        # allow more iterations
    C=1.0,                # adjust if needed (smaller values = stronger regularization)
)

clf_2.fit(X_train_tfidf, train_df["is_human"])

# evaluate
y_pred = clf_2.predict(X_val_tfidf)

y_val = val_df["is_human"]
report = classification_report(y_val, y_pred)

You can also check results here:

In [522]:
print(report)

              precision    recall  f1-score   support

           0       0.80      0.91      0.85      5349
           1       0.74      0.55      0.63      2674

    accuracy                           0.79      8023
   macro avg       0.77      0.73      0.74      8023
weighted avg       0.78      0.79      0.78      8023

